# Segment Anything (SAM) — Intro Inference Notebook

This notebook is the first experiment in `cv-research-lab`.

Goal:

- Run Meta AI's Segment Anything Model (SAM) in Google Colab
- Generate segmentation masks for an image
- Understand the basic inference workflow
- Begin documenting research-engineering observations

This is intentionally a first-pass notebook: get the model running, inspect outputs, and write down what you learn.


## 1. Check GPU

In Colab, go to:

`Runtime` → `Change runtime type` → `Hardware accelerator` → `GPU`


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
device


## 2. Install Dependencies

We install SAM directly from the official GitHub repository.

This may take a minute the first time you run it in Colab.


In [ ]:
!pip -q install git+https://github.com/facebookresearch/segment-anything.git
!pip -q install opencv-python matplotlib


## 3. Download a SAM Checkpoint

We start with **ViT-B**, the smallest standard SAM checkpoint. It is easier to run in Colab than the larger ViT-L or ViT-H models.


In [ ]:
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
!ls -lh sam_vit_b_01ec64.pth


## 4. Imports and Helper Functions

These helper functions visualize SAM's generated masks.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator


def show_image(image, title="Image", figsize=(8, 8)):
    """Display an RGB image."""
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()


def show_anns(anns):
    """Overlay SAM masks on the current matplotlib image."""
    if len(anns) == 0:
        return

    sorted_anns = sorted(anns, key=lambda x: x["area"], reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((
        sorted_anns[0]["segmentation"].shape[0],
        sorted_anns[0]["segmentation"].shape[1],
        4
    ))
    img[:, :, 3] = 0

    for ann in sorted_anns:
        m = ann["segmentation"]
        color_mask = np.concatenate([np.random.random(3), [0.35]])
        img[m] = color_mask

    ax.imshow(img)


def summarize_masks(masks, max_rows=10):
    """Print a compact summary of generated masks."""
    print(f"Number of masks: {len(masks)}")
    print("\nTop masks by area:")

    for i, mask in enumerate(sorted(masks, key=lambda x: x["area"], reverse=True)[:max_rows]):
        print(
            f"{i+1:02d}. area={mask['area']}, "
            f"predicted_iou={mask['predicted_iou']:.3f}, "
            f"stability_score={mask['stability_score']:.3f}"
        )


## 5. Load a Sample Image

For a first run, this notebook downloads a public sample image.

Later, replace this with your own image, ideally a street scene, driving scene, or object-rich image.


In [ ]:
# Public sample image
!wget -q -O sample_image.jpg https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/dog.jpg

image_bgr = cv2.imread("sample_image.jpg")

if image_bgr is None:
    raise FileNotFoundError("Image could not be loaded. Check the image path or download URL.")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
show_image(image_rgb, title="Input Image")


## 6. Load SAM

SAM has three main model sizes:

- `vit_b`: smallest and easiest to run
- `vit_l`: larger
- `vit_h`: largest and strongest, but more expensive

We use `vit_b` for this starter notebook.


In [ ]:
sam_checkpoint = "sam_vit_b_01ec64.pth"
model_type = "vit_b"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(sam)

print("SAM loaded on", device)


## 7. Generate Masks

The automatic mask generator proposes many possible object/region masks without requiring a prompt.

This is useful for initial exploration, but SAM can also use prompts such as points or boxes.


In [ ]:
masks = mask_generator.generate(image_rgb)
summarize_masks(masks)


## 8. Visualize Masks

The overlay below shows the regions SAM segmented in the image.


In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(image_rgb)
show_anns(masks)
plt.axis("off")
plt.title("SAM Automatic Masks")
plt.show()


## 9. Inspect Individual Masks

This helps you understand what SAM is producing beyond the overlay.


In [ ]:
# Sort masks from largest area to smallest
sorted_masks = sorted(masks, key=lambda x: x["area"], reverse=True)

# Change this index to inspect different masks
mask_index = 0
selected = sorted_masks[mask_index]["segmentation"]

plt.figure(figsize=(8, 8))
plt.imshow(selected, cmap="gray")
plt.title(f"Mask {mask_index} | Area: {sorted_masks[mask_index]['area']}")
plt.axis("off")
plt.show()


## 10. Research Notes

Use this section to document what you learn.

### Initial Observations

- What kinds of objects or regions did SAM segment well?
- What did it miss?
- Did it over-segment the image into too many pieces?
- Were the largest masks meaningful?
- How might this behave on driving scenes?

### Architecture Questions

- What does the image encoder produce?
- What does the mask decoder use as input?
- How does prompt-based segmentation differ from automatic mask generation?
- Why might a transformer architecture be useful for segmentation?

### Research Engineering Questions

- How long did inference take?
- How much GPU memory was used?
- What parameters might control the number or quality of masks?
- What would need to change to use this in an autonomous-vehicle perception pipeline?


## 11. Next Experiments

Possible follow-up experiments:

1. Try a driving-scene image from KITTI, nuScenes, or a dashcam image.
2. Compare automatic masks with point-prompt or box-prompt masks.
3. Change mask generator parameters and observe the effect.
4. Measure inference time.
5. Write a short summary of how SAM works.
6. Compare SAM output with semantic segmentation models.
